In [1]:
import os, math, time, torch.multiprocessing
import numpy as np, pandas as pd
import torch, torch.nn as nn, torchvision, torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import roc_auc_score

torch.multiprocessing.set_sharing_strategy("file_system")

np.random.seed(42)
torch.manual_seed(42)

torch.set_num_threads(os.cpu_count() or 1)

torch.backends.cudnn.allow_tf32 = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True  # already set, kept for completeness

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device




device(type='cuda')

In [2]:
ROOT_DIR = "/kaggle/input/ranzcr-clip-catheter-line-classification/test"
TRAIN_ROOT = "/kaggle/input/ranzcr-clip-catheter-line-classification/train"
SAMPLE_SUB_PATH = (
    "/kaggle/input/ranzcr-clip-catheter-line-classification/sample_submission.csv"
)
TRAIN_CSV_PATH = "/kaggle/input/ranzcr-clip-catheter-line-classification/train.csv"

sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
train_df = pd.read_csv(TRAIN_CSV_PATH)

TARGET_COLUMNS = [col for col in sample_sub.columns if col != "StudyInstanceUID"]

DEBUG = False
if DEBUG:
    BATCH_SIZE = 4
    EPOCHS = 2
else:
    BATCH_SIZE = 256 if torch.cuda.is_available() else 64
    EPOCHS = 3  # unchanged logic
IMG_SIZE = 256




In [3]:
import torchvision.io as io
import torchvision.transforms.functional as TF


class RanzcrClipTrainDataset(Dataset):
    """Dataset returning (image tensor, multi‑label vector) for training/validation."""

    def __init__(self, df, root_dir):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.ids = self.df["StudyInstanceUID"].values
        self.labels = self.df[TARGET_COLUMNS].values.astype(np.float32)

        self.mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        self.std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def __len__(self):
        return len(self.ids)

    def _load_image(self, uid):
        img_path = os.path.join(self.root_dir, f"{uid}.jpg")
        img = io.read_image(img_path)  # shape: (3, H, W), uint8
        img = img.float() / 255.0  # to float in [0,1]
        img = TF.resize(img, [IMG_SIZE, IMG_SIZE])
        img = (img - self.mean) / self.std  # normalize
        return img

    def __getitem__(self, idx):
        uid = self.ids[idx]
        image = self._load_image(uid)
        label = torch.from_numpy(self.labels[idx])
        return image, label




In [4]:
common_transforms = None  # not used; transforms are applied inside the dataset




In [5]:
full_dataset = RanzcrClipTrainDataset(train_df, TRAIN_ROOT)
val_size = int(0.1 * len(full_dataset))
train_size = len(full_dataset) - val_size
train_dataset, val_dataset = random_split(
    full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42)
)

# increased workers for faster I/O and larger prefetch to keep GPU fed
num_workers = min(8, os.cpu_count() or 1)
pin_mem = torch.cuda.is_available()  # pin_memory helps when using GPU

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=pin_mem,
    prefetch_factor=4,  # larger prefetch improves pipeline throughput
    persistent_workers=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_mem,
    prefetch_factor=4,
    persistent_workers=True,
)




/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [6]:
class CustomPretrainedModel(nn.Module):
    def __init__(self, num_targets):
        super().__init__()
        self.backbone = torchvision.models.densenet121(pretrained=True)
        n_features = self.backbone.classifier.in_features
        self.backbone.classifier = nn.Linear(n_features, num_targets)

    def forward(self, x):
        return self.backbone(x)


model = CustomPretrainedModel(num_targets=len(TARGET_COLUMNS)).to(device)

# Removed torch.compile to avoid one‑time compilation overhead while keeping identical logic
# model = torch.compile(model)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

use_amp = torch.cuda.is_available()
if use_amp:
    scaler = torch.cuda.amp.GradScaler()
else:
    scaler = None  # placeholder; not used on CPU




/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet121_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet121_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth
100%|██████████| 30.8M/30.8M [00:00<00:00, 114MB/s]
/tmp/ipykernel_55/3209043281.py:22: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.

In [7]:
print("Starting training...")
for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for imgs, labs in train_loader:
        imgs = imgs.to(device, non_blocking=True)
        labs = labs.to(device, non_blocking=True)

        optimizer.zero_grad()
        if use_amp:
            with torch.cuda.amp.autocast():
                logits = model(imgs)
                loss = criterion(logits, labs)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(imgs)
            loss = criterion(logits, labs)
            loss.backward()
            optimizer.step()

        epoch_loss += loss.item() * imgs.size(0)
    epoch_loss /= len(train_loader.dataset)
    print(f"Epoch {epoch}/{EPOCHS} – Train loss: {epoch_loss:.4f}")


def evaluate_auc(loader, net):
    net.eval()
    all_labels, all_preds = [], []
    with torch.no_grad():
        for imgs, labs in loader:
            imgs = imgs.to(device, non_blocking=True)
            if use_amp:
                with torch.cuda.amp.autocast():
                    logits = net(imgs)
                    probs = torch.sigmoid(logits).cpu().numpy()
            else:
                logits = net(imgs)
                probs = torch.sigmoid(logits).cpu().numpy()
            all_preds.append(probs)
            all_labels.append(labs.numpy())
    preds = np.vstack(all_preds)
    labels = np.vstack(all_labels)
    aucs = []
    for i in range(labels.shape[1]):
        try:
            aucs.append(roc_auc_score(labels[:, i], preds[:, i]))
        except ValueError:
            aucs.append(np.nan)
    return np.nanmean(aucs)


val_auc = evaluate_auc(val_loader, model)
print(f"Final validation macro AUC: {val_auc:.4f}")

print("Training completed.")




Starting training...


/tmp/ipykernel_55/2821059065.py:11: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 1/3 – Train loss: 0.3070
Epoch 2/3 – Train loss: 0.1893
Epoch 3/3 – Train loss: 0.1676


/tmp/ipykernel_55/2821059065.py:35: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Final validation macro AUC: 0.8679
Training completed.


In [8]:
class RanzcrClipTestDataset(Dataset):
    def __init__(self, ids, root_dir):
        self.ids = ids
        self.root_dir = root_dir
        self.mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        self.std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def __len__(self):
        return len(self.ids)

    def _load_image(self, uid):
        img_path = os.path.join(self.root_dir, f"{uid}.jpg")
        img = io.read_image(img_path).float() / 255.0
        img = TF.resize(img, [IMG_SIZE, IMG_SIZE])
        img = (img - self.mean) / self.std
        return img

    def __getitem__(self, idx):
        uid = self.ids[idx]
        return self._load_image(uid)


test_dataset = RanzcrClipTestDataset(
    ids=sample_sub["StudyInstanceUID"].values,
    root_dir=ROOT_DIR,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_mem,
    prefetch_factor=4,
    persistent_workers=True,
)
print(f"Test loader prepared, total batches: {len(test_loader)}")




Test loader prepared, total batches: 12


/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [9]:
def inference(net, loader):
    net.eval()
    all_probs = []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device, non_blocking=True)
            if use_amp:
                with torch.cuda.amp.autocast():
                    logits = net(batch)
                    probs = torch.sigmoid(logits)
            else:
                logits = net(batch)
                probs = torch.sigmoid(logits)
            all_probs.append(probs.cpu().numpy())
    return np.vstack(all_probs)


y_prob = inference(model, test_loader)
print(f"Inference completed, shape of probabilities: {y_prob.shape}")




/tmp/ipykernel_55/3204832523.py:8: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Inference completed, shape of probabilities: (3009, 9)


In [10]:
submission_df = pd.DataFrame(
    data=np.concatenate(
        [sample_sub["StudyInstanceUID"].values.reshape(-1, 1), y_prob], axis=1
    ),
    columns=sample_sub.columns,
)
submission_path = "./submission.csv"
submission_df.to_csv(submission_path, index=False)
print(f"Submission saved to {submission_path}")

Submission saved to ./submission.csv
